# SDE-Net paper-faithful t+1: analisi post-hoc con STGAN

Questo notebook non riaddestra SDE-Net o STGAN. Usa le predizioni del modello specializzato **t+1**, sostituisce le etichette MTGFlow con le decisioni STGAN sulla coppia esatta \(location, timestamp target\) e rigenera la stessa suite per bin del notebook principale.

Gli azzeramenti solari regionali isolati e l'ora immediatamente successiva vengono esclusi dall'analisi fisica; il **clean top-1% STGAN** viene ricalcolato sulle sole coordinate valide.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'physiq_pv').is_dir():
    for parent in ROOT.parents:
        if (parent / 'physiq_pv').is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from physiq_pv.experiments import sde_pipeline as pipe
from physiq_pv.reporting.pointwise_detector_posthoc import build_pointwise_detector_evaluation
print('repo root:', ROOT)

## 1. Percorsi e protocollo

In [ ]:
STGAN_SEED = 20
STGAN_SEED_DIR = Path(os.environ.get(
    'STGAN_SEED_DIR', ROOT / 'outputs' / 'pvgis_stgan' / 'paper_reference' / f'seed_{STGAN_SEED}'
)).resolve()
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'

PVGIS_2019 = Path(os.environ.get(
    'PVGIS_2019_FILE', ROOT / 'data' / 'pvgis' / 'piedmont_pvgis_2019.nc'
)).resolve()
STGAN_PREPARED_MANIFEST = (
    ROOT / 'outputs' / 'pvgis_stgan' / 'prepared' / 'manifest.csv'
).resolve()
PVGIS_QUALITY_CANDIDATES = (PVGIS_2019, STGAN_PREPARED_MANIFEST)
PVGIS_QUALITY_SOURCE = next(
    (path for path in PVGIS_QUALITY_CANDIDATES if path.is_file()), PVGIS_2019
)

BASE_CONFIG = {**pipe.DEFAULT_CONFIG,
    'name': 'paper_faithful_gaussian_detector_mtgflow_ep60',
    'horizon': 1,
    'epochs': 60, 'batch_size': 16, 'lr': 1e-4, 'lr_g': 1e-2,
    'dropout': 0.0, 'train_normal_only': False,
    'anomaly_source': 'detector', 'ood_smoke_test': True,
    'sde_sigma_warmup_epochs': 30, 'irradiance_loss_weight': 0.1,
    'detector_regional_quantile': 0.975,
}
assert BASE_CONFIG['horizon'] == 1
SDE_PREDICTIONS = Path(os.environ.get(
    'SDE_PREDICTIONS_T1', ROOT / pipe.make_out_dir(BASE_CONFIG) / 'predictions.csv'
)).resolve()
EVALUATION_DIR = Path(os.environ.get(
    'STGAN_POSTHOC_ROOT',
    ROOT / 'outputs' / f'sde_stgan_t1_paper_faithful_seed{STGAN_SEED}_quality_filtered'
)).resolve()

CLEAN_TOP_K_PERCENT = 1.0
MIN_MATCH_FRACTION = 0.90
RUN_RELABEL = not (EVALUATION_DIR / 'evaluation_source.json').is_file()
RUN_ANALYSIS = True
ALLOW_OVERWRITE = False

print('SDE-Net t+1:', SDE_PREDICTIONS)
print('STGAN:', STGAN_SCORES)
print('PVGIS quality source:', PVGIS_QUALITY_SOURCE)
print('output:', EVALUATION_DIR)

In [ ]:
missing = [
    path for path in (SDE_PREDICTIONS, STGAN_SCORES, PVGIS_QUALITY_SOURCE)
    if not path.is_file()
]
if missing:
    raise FileNotFoundError('File mancanti:\n' + '\n'.join(map(str, missing)))

stgan_header = set(pd.read_csv(STGAN_SCORES, nrows=0).columns)
required_stgan = {'location', 'timestamp', 'anomaly_score', 'is_anomaly'}
if not required_stgan <= stgan_header:
    raise ValueError(f'Colonne STGAN mancanti: {sorted(required_stgan - stgan_header)}')

sde_header = set(pd.read_csv(SDE_PREDICTIONS, nrows=0).columns)
required_sde = {'location', 'timestamp', 'y_true'}
if not required_sde <= sde_header or not ({'y_pred', 'y_pred_mean'} & sde_header):
    raise ValueError(f'Schema SDE-Net t+1 non compatibile: {sorted(sde_header)}')
if 'horizon_hours' in sde_header:
    horizons = set(pd.read_csv(SDE_PREDICTIONS, usecols=['horizon_hours'])['horizon_hours'].dropna().astype(int))
    if horizons != {1}:
        raise ValueError(f'Il CSV non è la run specializzata t+1: orizzonti {sorted(horizons)}')
print('OK: input SDE-Net t+1, STGAN e controllo qualità disponibili')

## 2. Relabel STGAN puntuale e filtro qualità

In [ ]:
if RUN_RELABEL:
    RELABEL_RESULT = build_pointwise_detector_evaluation(
        SDE_PREDICTIONS, STGAN_SCORES, EVALUATION_DIR,
        detector_name='stgan', min_match_fraction=MIN_MATCH_FRACTION,
        allow_overwrite=ALLOW_OVERWRITE,
        pvgis_quality_source=PVGIS_QUALITY_SOURCE,
        clean_top_k_percent=CLEAN_TOP_K_PERCENT,
    )
else:
    print('Output evaluation-only già presente:', EVALUATION_DIR)

metadata_path = EVALUATION_DIR / 'evaluation_source.json'
if not metadata_path.is_file():
    raise FileNotFoundError(metadata_path)
AUDIT = json.loads(metadata_path.read_text(encoding='utf-8'))
audit_columns = [
    'detector', 'forecast_mode', 'source_prediction_rows',
    'detector_matched_rows_before_quality_filter', 'matched_rows',
    'excluded_unmatched_rows', 'excluded_data_quality_rows', 'match_fraction',
    'solar_dropout_timestamps', 'data_quality_timestamps',
    'clean_top_k_percent', 'original_detector_anomalies',
    'clean_detector_anomalies', 'normal_rows', 'rare_rows',
]
display(pd.DataFrame([AUDIT])[[column for column in audit_columns if column in AUDIT]])

In [ ]:
QUALITY_ISSUES = pd.read_csv(EVALUATION_DIR / 'pvgis_data_quality_issues.csv')
display(Markdown('### Timestamp esclusi come problemi di qualità'))
display(QUALITY_ISSUES)

joined_path = EVALUATION_DIR / 'predictions.csv'
joined_header = set(pd.read_csv(joined_path, nrows=0).columns)
assert 'anomaly_group' in joined_header
assert 'event_group' not in joined_header
print('OK: le label MTGFlow sono state sostituite dalle label STGAN clean top-1%')

## 3. Stessa analisi per bin della pipeline t+1

In [ ]:
ANALYSIS_COMMAND = pipe.build_analysis_command(
    str(EVALUATION_DIR), BASE_CONFIG, predictions=str(joined_path)
)
if RUN_ANALYSIS:
    subprocess.run(ANALYSIS_COMMAND, check=True, cwd=ROOT)
elif not (EVALUATION_DIR / 'daytime_bin_anomaly_metrics.csv').is_file():
    raise FileNotFoundError(EVALUATION_DIR / 'daytime_bin_anomaly_metrics.csv')

FIGURE_PATHS = pipe.build_posthoc_figures(str(EVALUATION_DIR))
print(f'Figure generate: {len(FIGURE_PATHS)}')
for name, path in sorted(FIGURE_PATHS.items()):
    display(Markdown(f'### {name} — STGAN clean top 1%, t+1'))
    display(Image(filename=str(path)))

In [ ]:
BIN_METRICS = pd.read_csv(EVALUATION_DIR / 'daytime_bin_anomaly_metrics.csv')
COPY_REPORT = EVALUATION_DIR / 'stgan_t1_bin_metrics_copy_report.csv'
BIN_METRICS.to_csv(COPY_REPORT, index=False)
display(BIN_METRICS)
print('BEGIN_STGAN_T1_BIN_METRICS_CSV')
print(BIN_METRICS.to_csv(index=False), end='')
print('END_STGAN_T1_BIN_METRICS_CSV')
print('File:', COPY_REPORT)

## Interpretazione

Il modello di forecasting resta esattamente quello paper-faithful specializzato t+1. Cambiano esclusivamente le etichette di valutazione: normale e raro sono stabiliti dal clean top-1% STGAN sullo stesso target. In questo modo il confronto con la post-hoc MTGFlow non è confuso dal training multi-orizzonte.